# Batch Operating Mode Test Notebook

This notebook tests the **BATCH** operating mode implementation.

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Batch-specific validations

## 1. Setup and Imports

In [23]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.gas_controllers import GasController
from animakinetica.models.stoichiometry import StoichiometryMode

print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [24]:
# Load base parameters
# params = SimulationParameters.load_parameters()
# params.update("simple_kinetics")

params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI"])
params.update("excess_kinetics_adjusted")
# Initialize simulation with BATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    stoichiometry_mode=StoichiometryMode.CONSTANT,
    operating_mode=OperatingMode.BATCH,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,  # Store detailed data for testing
    solver_method="BDF",
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Stoichiometry Mode: {sim.stoichiometry_mode}")
print(f"  Gas Controller: {sim.gas_controller}")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).
✓ Simulation initialized
  Operating Mode: Batch
  Kinetics Model: Stoichiometric
  Stoichiometry Mode: StoichiometryMode.CONSTANT
  Gas Controller: GasController.CASCADE_N_FG_XO2


## 4. Multi-Batch Run

In [25]:
# Run 5 repeated batches
num_batches = 5
multi_result = sim.run_simulation(number_of_batches=5)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.2f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L")

run_simulation execution time: 225.103 ms
✓ 5 batches completed

Summary Statistics:
  Average batch duration: 95.75 h
  STY: 0.35 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 97.05 h, Xv=1.04e+10 cells/L
  Batch 2: 95.50 h, Xv=9.82e+09 cells/L
  Batch 3: 95.40 h, Xv=9.79e+09 cells/L
  Batch 4: 95.40 h, Xv=9.79e+09 cells/L
  Batch 5: 95.40 h, Xv=9.79e+09 cells/L


## 5. Data Integrity Validation

In [26]:
# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("Termination reason:", batch_1.termination_reason)

print("\n1. State Variables:")
# Explicit index mapping matches y_points array: [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA, C_Osm_bg]
state_var_indices = {
    'Xv': 0, 'C_Glc': 1, 'C_Gln': 2, 'C_Lac': 3, 'C_Amm': 4,
    'V': 5, 'C_O2': 6, 'C_CT': 7, 'C_H': 8, 'C_Na': 9, 'C_AA': 10,
    'C_Osm_bg': 11
}
for var, idx in state_var_indices.items():
    data = np.asarray(batch_1.y_points[idx], dtype=float)
    print(f"  {var} (idx {idx}): {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_O2', 'q_CO2', 'q_AA']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n✓ All data integrity checks passed")


Data Integrity Checks for Batch 1:
Termination reason: Aa depletion

1. State Variables:
  Xv (idx 0): 121 points, range [6.00e+08, 1.04e+10]
  C_Glc (idx 1): 121 points, range [7.70e+00, 2.50e+01]
  C_Gln (idx 2): 121 points, range [4.92e+00, 1.00e+01]
  C_Lac (idx 3): 121 points, range [0.00e+00, 1.01e+01]
  C_Amm (idx 4): 121 points, range [0.00e+00, 3.32e+00]
  V (idx 5): 121 points, range [2.00e+03, 2.00e+03]
  C_O2 (idx 6): 121 points, range [1.00e-02, 1.00e-02]
  C_CT (idx 7): 121 points, range [1.75e+01, 2.90e+01]
  C_H (idx 8): 121 points, range [6.31e-08, 6.31e-08]
  C_Na (idx 9): 121 points, range [0.00e+00, 0.00e+00]
  C_AA (idx 10): 121 points, range [1.00e-03, 5.00e+00]
  C_Osm_bg (idx 11): 121 points, range [2.50e+02, 2.50e+02]

2. Kinetic Rates:
  mu: 121 points, range [2.46e-02, 3.03e-02]
  q_Glc: 121 points, range [4.34e-11, 5.35e-11]
  q_Gln: 121 points, range [1.27e-11, 1.57e-11]
  q_Lac: 121 points, range [2.53e-11, 3.12e-11]
  q_Amm: 121 points, range [8.35e-12, 1

## 6. Performance Metrics

In [27]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Conversion Efficiency: {multi_result.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {multi_result.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {multi_result.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_conversion_efficiency_m3_kg'] < 1000, "Medium conversion efficiency should be positive and reasonable"
assert multi_result.summary_stats['protein_conversion_efficiency_kg_kg'] > 0, "Protein conversion efficiency should be positive"
assert multi_result.summary_stats['feed_conversion_efficiency_kg_kg'] > 0, "Feed conversion efficiency should be positive"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")


Performance Metrics (5-Batches):

 Total Biomass: 14.15 kg
  STY: 0.355 kg/m3/day
  Medium Conversion Efficiency: 0.672 m3/kg DCM
  Protein Conversion Efficiency: 2.190 kg/kg
  Feed Conversion Efficiency: 4.390 kg/kg
  Total Simulation Time: 478.75 h
  Number of Batches: 5

✓ Performance metrics validated


## 7. Batch-Specific Validations

In [28]:
print("Batch-Specific Validations:")

# 1. Volume should remain constant (no feed/harvest)
print("\n1. Volume Constancy:")
for i, batch in enumerate(sim.data.batches, 1):
    V_initial = batch.y_points[5][0]
    V_final = batch.y_points[5][-1]
    V_change = abs(V_final - V_initial)
    print(f"  Batch {i}: V_initial={V_initial:.4f} L, V_final={V_final:.4f} L, |ΔV|={V_change:.6f} L")
    assert V_change < (0.01*V_final), f"Batch {i}: Volume changed by {V_change} L which is more than 1% of final volume (should be close to constant except for pH corrections)"

print("  ✓ Volume remains close to constant in all batches, with minor variations due to pH corrections")

# 2. Substrate depletion check
print("\n2. Substrate Depletion:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Glc_final = batch.y_points[1][-1]
    C_Gln_final = batch.y_points[2][-1]
    C_AA_final  = batch.y_points[10][-1]
    print(f"  Batch {i}: C_Glc_final={C_Glc_final:.3f} mmol/L, "
          f"C_Gln_final={C_Gln_final:.3f} mmol/L, "
          f"C_AA_final={C_AA_final:.3f} mmol/L")

# 3. Cell growth dynamics
print("\n3. Cell Growth Dynamics:")
for i, batch in enumerate(sim.data.batches, 1):
    Xv_initial = batch.y_points[0][0]
    Xv_final = batch.y_points[0][-1]
    Xv_max = batch.y_points[0].max()
    fold_increase = Xv_max / Xv_initial
    print(f"  Batch {i}: Xv_max={Xv_max:.2e} cells/L, Fold increase={fold_increase:.2f}x")
    assert Xv_max > Xv_initial, f"Batch {i}: No cell growth detected"

print("  ✓ Cell growth observed in all batches")

# 4. Metabolite accumulation
print("\n4. Metabolite Accumulation:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Lac_initial = batch.y_points[3][0]
    C_Lac_final = batch.y_points[3][-1]
    C_Amm_initial = batch.y_points[4][0]
    C_Amm_final = batch.y_points[4][-1]
    print(f"  Batch {i}: Lactate: {C_Lac_initial:.2f} → {C_Lac_final:.2f} mmol/L")
    print(f"           Ammonia: {C_Amm_initial:.2f} → {C_Amm_final:.2f} mmol/L")
    assert C_Lac_final >= C_Lac_initial, f"Batch {i}: Lactate decreased (unphysical)"
    assert C_Amm_final >= C_Amm_initial, f"Batch {i}: Ammonia decreased (unphysical)"

print("  ✓ Metabolites accumulate as expected")

# 5. Amino acid consumption check
print("\n5. Amino Acid Consumption:")
for i, batch in enumerate(sim.data.batches, 1):
    C_AA_initial = batch.y_points[10][0]
    C_AA_final = batch.y_points[10][-1]
    C_AA_consumed = C_AA_initial - C_AA_final
    print(f"  Batch {i}: C_AA: {C_AA_initial:.2f} → {C_AA_final:.2f} mmol/L  "
          f"(consumed: {C_AA_consumed:.2f} mmol/L)")
    assert C_AA_final <= C_AA_initial, f"Batch {i}: Amino acids increased (unphysical — no feed in BATCH mode)"

print("  ✓ Amino acids consumed as expected in all batches")

print("\n✓ All batch-specific validations passed")


Batch-Specific Validations:

1. Volume Constancy:
  Batch 1: V_initial=2000.0000 L, V_final=2000.0000 L, |ΔV|=0.000000 L
  Batch 2: V_initial=2000.0000 L, V_final=2000.0000 L, |ΔV|=0.000000 L
  Batch 3: V_initial=2000.0000 L, V_final=2000.0000 L, |ΔV|=0.000000 L
  Batch 4: V_initial=2000.0000 L, V_final=2000.0000 L, |ΔV|=0.000000 L
  Batch 5: V_initial=2000.0000 L, V_final=2000.0000 L, |ΔV|=0.000000 L
  ✓ Volume remains close to constant in all batches, with minor variations due to pH corrections

2. Substrate Depletion:
  Batch 1: C_Glc_final=7.699 mmol/L, C_Gln_final=4.924 mmol/L, C_AA_final=0.001 mmol/L
  Batch 2: C_Glc_final=7.699 mmol/L, C_Gln_final=4.924 mmol/L, C_AA_final=0.001 mmol/L
  Batch 3: C_Glc_final=7.699 mmol/L, C_Gln_final=4.924 mmol/L, C_AA_final=0.001 mmol/L
  Batch 4: C_Glc_final=7.699 mmol/L, C_Gln_final=4.924 mmol/L, C_AA_final=0.001 mmol/L
  Batch 5: C_Glc_final=7.699 mmol/L, C_Gln_final=4.924 mmol/L, C_AA_final=0.001 mmol/L

3. Cell Growth Dynamics:
  Batch 1: X

## 8. Visualization


In [29]:
from animakinetica.analysis.batch_visualization import plot_batch_biomass_metabolites
from animakinetica.analysis.process_visualization import plot_simulation_profiles

# Biomass, metabolites, and amino acids — 3-panel plot for batch 1
fig_batch = plot_batch_biomass_metabolites(sim, batch_number=1)
fig_batch.show()


In [30]:
# Multi-batch profile overlay — includes amino acids profile (fig index 5)
figs = plot_simulation_profiles(sim, max_batches=num_batches)

profile_labels = [
    "Cell Density", "Glucose", "Glutamine", "Lactate", "Ammonia",
    "Amino Acids", "Dissolved O₂", "Total Dissolved Carbon",
    "pH", "Dissolved CO₂", "Osmolality", "Na⁺ Concentration"
]

print(f"Generated {len(figs)} profile figures:")
for i, label in enumerate(profile_labels[:len(figs)]):
    print(f"  figs[{i}] — {label}")

# Show amino acids profile
figs[0].show()


Generated 12 profile figures:
  figs[0] — Cell Density
  figs[1] — Glucose
  figs[2] — Glutamine
  figs[3] — Lactate
  figs[4] — Ammonia
  figs[5] — Amino Acids
  figs[6] — Dissolved O₂
  figs[7] — Total Dissolved Carbon
  figs[8] — pH
  figs[9] — Dissolved CO₂
  figs[10] — Osmolality
  figs[11] — Na⁺ Concentration


In [31]:
figs_dict = sim.data.plot_all(1,None)

for fig in figs_dict.values():
    fig.show()
    



Generating all plots for Batch...

[1/16] Plotting batch 1: Biomass & Metabolites...
[2/16] Plotting batch 1: Volume & Feed...
[3/16] Plotting batch 1: Gas Transfer...
[4/16] Plotting batch 1: Nutrient Factors...
[5/16] Plotting batch 1: Inhibition Factors...
[6/16] Plotting batch 1: Kinetic Rates...
[7/16] Plotting batch 1: Complete Kinetics Overview...
[8/16] Plotting batch 1: Stoichiometric Yields...
[9/16] Plotting cumulative harvest...
[10/16] Plotting cumulative medium use...
[11/16] Plotting cumulative gas use...
[12-16/16] Plotting simulation profiles...

✓ All 16 plots generated successfully!

